# PHerc0800 — First Letters ink sweep

Runs the **exact ink stage of the winning PHerc0343 submission** (Erwin Nieuwlaar, $50K,
9 Oct 2026) against the **already-published 31-layer surface volumes of PHerc0800**, an
eligible First Letters scroll on which no ink has been found.

Why this is cheap: PHerc0800 is the only eligible scroll with published segments *rendered
on its eligible volume* (`20250521135224`). Spiral fitting, snapping, flattening and
rendering — days of GPU work, and the entire geometry bottleneck — are already done and
public. Only inference is left.

What is new versus what has already been tried on these renders: four models averaged as
z-scores over a depth sweep, with one contrast stretch per model fixed across all depths.
Single checkpoints have surely been run on these public renders; that ensemble is what
turned a null result into letters on PHerc0343.

### Before running

1. **Settings -> Accelerator -> GPU T4 x2**, and **Internet -> On**.
   Internet needs a phone-verified Kaggle account; without it there is no S3 and no pip.
2. **Keep this notebook private.** Prize terms: *you must not make your discovery public
   until the prize is officially announced*. A public notebook publishes its outputs.
3. Expect roughly 2 h for d = -3..+3 on one T4. The 12 h session limit is the real
   constraint, so the sweep is resumable: maps that already exist are skipped.


## 1. Check the machine

The driver version decides the torch build. villa's bootstrap defaults to CUDA 13 wheels,
which need a recent driver; Kaggle's is usually older, so pick the backend from what the
driver actually reports.


In [ ]:
!nvidia-smi
import subprocess
drv = subprocess.run(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
major = int(float(drv.split('.')[0]))
BACKEND = 'cu130' if major >= 580 else ('cu128' if major >= 525 else 'cu126')
print('driver', drv, '-> torch backend', BACKEND)


## 2. Set up villa, the checkpoints and the winning scripts

`setup_gpu.sh` pins villa to commit `0fb38c45` — the commit every ink map of the winning
submission was computed with — installs CPython 3.14 (the `vesuvius` package pins
`>=3.14,<3.15`), fetches the four public checkpoints with sha256 verification, and patches
the ensemble step to accept 31-layer renders instead of 66.

Work lives in `/kaggle/temp` (large, ephemeral). Only the final small PNGs go to
`/kaggle/working`.


In [ ]:
import os
os.environ['ROOT'] = ROOT = '/kaggle/temp/data'
os.environ['TORCH_BACKEND'] = BACKEND
!mkdir -p {ROOT}
!git clone -q https://github.com/Ayyyn/ink-det.git /kaggle/temp/ink-det 2>/dev/null || (cd /kaggle/temp/ink-det && git pull -q)
!bash /kaggle/temp/ink-det/sprint/setup_gpu.sh {ROOT} 2>&1 | tail -40


## 3. Pull one published render

The largest segment first: 2580 x 2580 px at 8.64 um = 4.97 cm2 of canvas, about half of
it covered by mesh. 4 cm2 is the *maximum* box the ten letters must fit inside, not a
coverage minimum, so this is comfortably large enough.

Level 0 only: about 200 MB, uncompressed uint8.


In [ ]:
SEG = '20251028222030'   # 2580x2580, 4.97 cm2 canvas - the biggest of the six
PY = ROOT + '/venv/bin/python'
!{PY} /kaggle/temp/ink-det/sprint/pull_render.py --scroll PHerc0800 --segment {SEG} --out {ROOT}/renders
!du -sh {ROOT}/renders/{SEG}.zarr


## 4. Look at the render before running any model

The challenge's own guidance: ink is sometimes visible directly in a flattened render,
usually as bright areas, and that alone qualifies. Always spend the thirty seconds.

Already checked on a laptop for all five usable PHerc0800 segments: crisp crossed fibers
everywhere, which means good geometry, and no visible text; the dark features are damage
voids. This cell keeps that check in the record rather than as an assumption.


In [ ]:
import numpy as np, zarr
import matplotlib.pyplot as plt
g = zarr.open(ROOT + '/renders/' + SEG + '.zarr', mode='r')
a = g['0'] if '0' in g else g
print('shape', a.shape, a.dtype)
mid = a[a.shape[0] // 2]
m = mid > 0
lo, hi = np.percentile(mid[m].astype(np.float32), [1, 99.5])
plt.figure(figsize=(11, 11))
plt.imshow(np.clip((mid - lo) / (hi - lo), 0, 1) * m, cmap='gray')
plt.axis('off'); plt.title(SEG + ' central layer (d=0)'); plt.show()


## 5. The sweep

Four models x every depth in the range x both faces of the sheet. On an unread scroll we
do not know which face carries the ink, so running both is not optional.

31 layers admit d = -7..+7 (the villa models take a 17-layer window s..s+16 centred on
layer 15 + d). Start at d = -3..+3: the central depths are where ink sits if the snap is
good, and a narrow first pass answers in about 2 h instead of 5. Widen afterwards.

`P_VILLA=1 P_HEC=1` because a T4 has 16 GB; the winning runs used 2+2 on 24 GB.


In [ ]:
os.environ['P_VILLA'] = '1'
os.environ['P_HEC'] = '1'
!bash /kaggle/temp/ink-det/sprint/sweep31.sh {ROOT}/renders/{SEG}.zarr {ROOT}/maps/{SEG} -3 3 2>&1 | tail -25
!ls {ROOT}/maps/{SEG} | head -20
!cat {ROOT}/maps/{SEG}/logs/timings.tsv 2>/dev/null | head -20


## 6. Ensemble, then look

One contrast stretch per model across every depth computed, then the four z-scores
averaged. The `_blind` variants are the other face of the sheet: a real text row appears
on one face only, so a feature that shows equally on both is an artefact, not ink.


In [ ]:
!bash /kaggle/temp/ink-det/sprint/ensemble31.sh {ROOT}/renders/{SEG}.zarr {ROOT}/maps/{SEG} {ROOT}/images/{SEG} '-3..3' '-1 0 1' 2>&1 | tail -20
import glob
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
shown = sorted(glob.glob(ROOT + '/images/' + SEG + '/*ensemble*.png'))
print(len(shown), 'ensemble images')
for p in shown:
    im = Image.open(p)
    w, h = im.size
    f = max(w, h) / 1400.0
    plt.figure(figsize=(14, 14))
    plt.imshow(im.resize((int(w / f), int(h / f))), cmap='gray')
    plt.axis('off'); plt.title(p.split('/')[-1]); plt.show()


## 7. Keep the results

PNGs and the stats JSON are small; the maps and the render are large and regenerable.
Save a notebook version to persist `/kaggle/working`.

If there is ink: **stop and submit before publishing anything.** If there is not, the next
segment is one line — change `SEG` — and the next scroll is PHerc0268 or PHerc1218, which
need the full spiral chain on a 24 GB box.


In [ ]:
!mkdir -p /kaggle/working/{SEG}
!cp {ROOT}/images/{SEG}/*.png /kaggle/working/{SEG}/ 2>/dev/null
!cp {ROOT}/images/{SEG}/*.json /kaggle/working/{SEG}/ 2>/dev/null
!cp {ROOT}/maps/{SEG}/logs/timings.tsv /kaggle/working/{SEG}/ 2>/dev/null
!du -sh /kaggle/working/{SEG}; ls /kaggle/working/{SEG}
